# Day 7 — HTTP & the `requests` Library

> ⚠️ **Why this matters.** Today your english-helper stops being a closed system and starts talking to the internet. Every modern app — Instagram, Spotify, ChatGPT, anything you've used today — is built on the same `GET` and `POST` calls you're about to write.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/07-http-requests.ipynb)

## What you'll do today

**Time:** 90 min lesson + 60 min mini-project + 45 min quiz.

By the end:

- [ ] You understand HTTP requests and responses at a working level
- [ ] You can use `requests` to make GET and POST calls
- [ ] You can read response status, headers, body
- [ ] You can handle network failures gracefully
- [ ] You've made your first call to the Free Dictionary API

## The mental model

Every web interaction is a **request** from your computer to a server, followed by a **response** back.

```
YOU ─── GET /entries/en/thorough HTTP/1.1 ───▶  api.dictionaryapi.dev
    ◀── HTTP/1.1 200 OK + JSON body         ───
```

Both messages have: a status line, headers (metadata), and an optional body. You'll spend your career reading and writing these.

> 💡 **In the wild:** Every API call to GitHub, OpenAI, Stripe, your bank — all just HTTP. Same protocol, same structure. Learn it once.

## 1. The HTTP methods you'll actually use

| Method | Used for | Body? |
|--------|----------|-------|
| `GET` | Fetch data — should not change anything on the server | No |
| `POST` | Create something new on the server | Yes |
| `PUT` | Replace an entire resource | Yes |
| `PATCH` | Update some fields of a resource | Yes |
| `DELETE` | Remove a resource | No |

**Today we mostly use `GET`** — fetching dictionary definitions. POST shows up Day 9.

## 2. The status codes you'll memorize

| Code | Meaning |
|------|---------|
| `200` | OK — happy path |
| `201` | Created — used for POST success |
| `204` | No content — successful, nothing to return |
| `400` | Bad Request — your request is malformed |
| `401` | Unauthorized — you need to log in / send a key |
| `403` | Forbidden — you're logged in but not allowed |
| `404` | Not Found — that resource doesn't exist |
| `429` | Too Many Requests — you're rate-limited; back off |
| `500` | Internal Server Error — they broke, not you |
| `503` | Service Unavailable — temporary overload |

**Memorize them.** When debugging API calls, the status code is the first clue.

> 💡 **2xx success. 3xx redirects. 4xx your fault. 5xx their fault.** That's the entire mental model.

## 3. Installing `requests`

In [ ]:
# In your english-helper project:
# $ uv add requests
#
# Then:
import requests
print(requests.__version__)

`requests` isn't in the standard library — it's so good everyone uses it anyway. The stdlib alternative is `urllib`, which works but is clunky.

## 4. Your first GET — the Free Dictionary API

In [ ]:
import requests

url = 'https://api.dictionaryapi.dev/api/v2/entries/en/thorough'
response = requests.get(url)

print('Status:', response.status_code)
print('Content-Type:', response.headers['content-type'])
print('Body length:', len(response.text), 'chars')

The `response` object has everything you need:

- `response.status_code` — int, the HTTP status
- `response.headers` — dict-like, the response headers
- `response.text` — raw body as a string
- `response.json()` — parse body as JSON, returns dict/list
- `response.ok` — True iff `200 <= status_code < 400`

## 5. Parsing the JSON response

In [ ]:
data = response.json()
print(type(data))      # list (this API returns a list of entries)
print(len(data))       # 1 (one entry)
entry = data[0]
print(entry.keys())    # word, phonetic, phonetics, meanings, etc.
print(entry['word'])   # 'thorough'

Dig deeper:

In [ ]:
for ph in entry['phonetics']:
    if 'text' in ph and 'audio' in ph and ph['audio']:
        print(f"IPA: {ph['text']}")
        print(f"Audio: {ph['audio']}")

> 💡 **Why APIs are good:** the Free Dictionary API was built by linguists. It has 470,000+ words with real pronunciations, IPA, meanings, examples, audio. You'd never type all this yourself.

## 6. Handling failures

In [ ]:
import requests

def safe_get(url: str) -> dict | None:
    try:
        response = requests.get(url, timeout=5)
    except requests.Timeout:
        print(f'Timeout fetching {url}')
        return None
    except requests.ConnectionError:
        print(f'No internet?')
        return None

    if response.status_code == 404:
        return None  # word not found
    response.raise_for_status()  # raise on other 4xx/5xx
    return response.json()

**Three rules for network calls:**

1. **Always set a `timeout=`.** Without it, your script can hang forever if the server is slow.
2. **Always check `status_code` or use `raise_for_status()`.** A 404 looks successful (no exception) until you try to use the data.
3. **Catch `requests.Timeout` and `requests.ConnectionError`.** They happen.

> ⚠️ **Don't catch `Exception`.** That hides bugs. Catch the specific exceptions you can handle.

## 7. POST requests (preview)

In [ ]:
# Example with httpbin.org — a request-mirroring service for testing
# requests.post('https://httpbin.org/post', json={'word': 'thorough', 'study': True})
# Returns a dict echoing your request back
# We won't use POST in english-helper much; you'll use it heavily in Phase 3 (your own API).

## End-of-day mini-project — `api.py`

> 🎯 **Today's piece:** wrap the Free Dictionary API in a clean module that english-helper can use.

### What you're building

A module `src/english_helper/api.py` exporting:

```python
def fetch_word(word: str) -> dict | None:
    '''Fetch word data from Free Dictionary API.
    Returns the raw API response (dict) or None if not found.
    '''

def extract_ipa(api_data: dict) -> str | None:
    '''Pull the IPA out of the API response, or None if not present.'''

def extract_definition(api_data: dict, max_chars: int = 200) -> str:
    '''Pull the first definition.'''

def extract_audio_url(api_data: dict) -> str | None:
    '''Pull the audio pronunciation URL, or None.'''
```

### Requirements

- Use `requests` with a 5-second timeout.
- Handle 404 → return None.
- Handle network errors → print a friendly message, return None.
- All functions type-hinted.
- At the bottom, `if __name__ == '__main__':` that fetches `'thorough'` and prints everything.

### Verify

```bash
$ uv run python -m english_helper.api
Fetching 'thorough'...
IPA: /ˈθʌrə/
Audio: https://api.dictionaryapi.dev/media/pronunciations/en/thorough-uk.mp3
Definition: complete with regard to every detail; not superficial or partial.
```

### Stretch
- Try a word that doesn't exist (`'asdfasdf'`). Confirm it prints "Not found" not crashes.
- Add a `download_audio(url, path)` helper that fetches the .mp3 file.
- Open the audio URL in the system's default media player when run.


## Connect to the project

> 🎯 **Connects to the project:** Tomorrow (Day 8) you'll cache these API responses to disk so you don't re-hit the API for words you've looked up before. Then Day 9 you'll integrate it into the CLI: `english-helper add thorough` becomes one command that fetches everything and adds it to your vocabulary.

## Self-check

<details>
<summary>1. What's the status code for "resource not found"?</summary>

`404`.
</details>

<details>
<summary>2. Why always set a <code>timeout=</code> on requests?</summary>

Without it, your script can hang indefinitely if the server is slow or down. Even a 30-second wait is too long for users.
</details>

<details>
<summary>3. What does <code>response.json()</code> return?</summary>

The parsed body — usually a dict or a list — assuming the body is valid JSON. If it isn't, you get a `json.JSONDecodeError`.
</details>

**Quiz:** [07-http-requests-quiz.ipynb](07-http-requests-quiz.ipynb)